In [ ]:
# Example 1: Verify the primary public API surface is importable.
from httpx_aiohttp import AiohttpTransport

print(f"AiohttpTransport = {AiohttpTransport}")
print(f"AiohttpTransport module = {AiohttpTransport.__module__}")

# AiohttpTransport must be an httpx async transport.
import httpx
assert issubclass(AiohttpTransport, httpx.AsyncBaseTransport), (
    f"AiohttpTransport should subclass httpx.AsyncBaseTransport, got bases {AiohttpTransport.__mro__}"
)
print("AiohttpTransport is a subclass of httpx.AsyncBaseTransport")


In [ ]:
# Example 2: Construct an AiohttpTransport with an aiohttp ClientSession factory.
# The `client` param takes a ClientSession instance or a SYNC zero-arg factory
# returning one (the transport calls it without awaiting).
import aiohttp
from httpx_aiohttp import AiohttpTransport

def _make_session() -> aiohttp.ClientSession:
    return aiohttp.ClientSession()

transport = AiohttpTransport(client=_make_session)
print(f"transport = {transport!r}")
print(f"transport type = {type(transport).__name__}")

# Clean up: transport supports async close via aclose().
assert hasattr(transport, "aclose"), "AiohttpTransport must expose aclose()"
await transport.aclose()
print("transport.aclose() completed")

In [ ]:
# Example 3: Wire AiohttpTransport into an httpx.AsyncClient and verify configuration.
import aiohttp
import httpx
from httpx_aiohttp import AiohttpTransport

def _session_factory() -> aiohttp.ClientSession:
    return aiohttp.ClientSession()

transport = AiohttpTransport(client=_session_factory)
async with httpx.AsyncClient(transport=transport) as client:
    print(f"client = {client!r}")
    print(f"client._transport = {type(client._transport).__name__}")
    assert isinstance(client._transport, AiohttpTransport)

print("AsyncClient successfully wraps AiohttpTransport")

In [ ]:
# Example 4: Inspect the package's public symbols + ensure AiohttpTransport is exposed.
import httpx_aiohttp

public_names = [n for n in dir(httpx_aiohttp) if not n.startswith("_")]
print(f"Public symbols ({len(public_names)}): {public_names}")

assert "AiohttpTransport" in public_names, "AiohttpTransport must be a top-level export"

# The transport class should accept httpx-style transport kwargs.
import inspect
sig = inspect.signature(httpx_aiohttp.AiohttpTransport.__init__)
param_names = list(sig.parameters)
print(f"AiohttpTransport.__init__ params: {param_names}")
assert "client" in param_names, "AiohttpTransport.__init__ must accept a 'client' param"


In [ ]:
# Example 5: End-to-end request httpx -> AiohttpTransport -> aiohttp against an
# in-process aiohttp server (self-contained: no external network dependence).
import aiohttp
from aiohttp import web
import httpx
from httpx_aiohttp import AiohttpTransport

async def handler(request: web.Request) -> web.Response:
    return web.json_response({"ok": True, "via": "aiohttp"})

app = web.Application()
app.router.add_get("/", handler)
runner = web.AppRunner(app)
await runner.setup()
site = web.TCPSite(runner, "127.0.0.1", 0)
await site.start()
host, port = runner.addresses[0][:2]

transport = AiohttpTransport(client=lambda: aiohttp.ClientSession())
async with httpx.AsyncClient(transport=transport, timeout=10.0) as client:
    response = await client.get(f"http://{host}:{port}/")
    print(f"status = {response.status_code}, body = {response.json()}")
    assert response.status_code == 200
    assert response.json() == {"ok": True, "via": "aiohttp"}

await runner.cleanup()
print("end-to-end httpx -> AiohttpTransport -> aiohttp round trip succeeded")